# RAG 기반 인재 추천

프로젝트 요구사항 텍스트를 입력받아 LLM이 포지션별로 파싱하고, 벡터 검색으로 적합한 엔지니어를 추천합니다.

In [1]:
from _bootstrap import setup_project_path

setup_project_path()

WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [2]:
import hashlib
import json
from pathlib import Path

from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_llm, create_retrieval_pipeline
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEERS

config = RetrievalConfig.from_env().with_overrides(chunk_size=2000, chunk_overlap=0, top_k=3)
ingest_service, search_service = create_retrieval_pipeline(config)
llm = create_llm(config)

# 데이터 해시 비교 → 변경 시에만 재임베딩
HASH_FILE = Path("../.ingest_hash")
current_hash = hashlib.md5(json.dumps(SAMPLE_ENGINEERS, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
stored_hash = HASH_FILE.read_text().strip() if HASH_FILE.exists() else ""

if current_hash != stored_hash:
    print("데이터 변경 감지 → 재임베딩 시작...")
    ingest_service.add_texts(
        texts=[item["text"] for item in SAMPLE_ENGINEERS],
        metadatas=[item["metadata"] for item in SAMPLE_ENGINEERS],
    )
    HASH_FILE.write_text(current_hash)
    print(f"완료. 해시: {current_hash}")
else:
    print(f"데이터 변경 없음 → 임베딩 스킵 (해시: {current_hash})")

데이터 변경 감지 → 재임베딩 시작...
완료. 해시: 4ce2104ca0f164263ebeb26f03ffba4f


In [3]:
# 사용자 입력 (프로젝트 요구사항 텍스트)
project_requirement = """
[프로젝트 명]
SK ITSM 이전 서비스

[프로젝트 개요]
기존 운영되던 ITSM 서비스 중 3개의 서비스를 마이그레이션

[총 인원 수]
3명

[요구인력]
 - 포지션: PL / 역할 : 백엔드 개발자 / 등급 : 고급, 특급 / 스킬 : Java, Spring / 인원 1명
 - 포지션 : 개발자 / 역할 : 프론트 / 등급 : 중급 / 스킬 : React / 인원 2명
"""

print(project_requirement)


[프로젝트 명]
SK ITSM 이전 서비스

[프로젝트 개요]
기존 운영되던 ITSM 서비스 중 3개의 서비스를 마이그레이션

[총 인원 수]
3명

[요구인력]
 - 포지션: PL / 역할 : 백엔드 개발자 / 등급 : 고급, 특급 / 스킬 : Java, Spring / 인원 1명
 - 포지션 : 개발자 / 역할 : 프론트 / 등급 : 중급 / 스킬 : React / 인원 2명



In [6]:
import json

# LLM으로 요구사항 파싱 → 포지션별 구조화
parse_prompt = f"""
아래 프로젝트 요구사항에서 포지션별 요구 인력을 JSON 배열로 추출하세요.

등급 변환 규칙:
- 특급 → EXPERT
- 고급 → SENIOR
- 중급 → INTERMEDIATE
- 초급 → JUNIOR

출력 형식 (JSON 배열만 출력, 설명 없이):
[
  {{
    "position": "포지션명",
    "role": "역할",
    "grades": ["SENIOR"],
    "skills": ["Java", "Spring"],
    "count": 1,
    "search_query": "벡터 검색에 사용할 자연어 쿼리"
  }}
]

프로젝트 요구사항:
{project_requirement}
"""

response = llm.invoke(parse_prompt)
content = response.content
if isinstance(content, list):
    content = "".join(block if isinstance(block, str) else block.get("text", "") for block in content)
raw = content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
positions = json.loads(raw)
positions

[{'position': 'PL',
  'role': '백엔드 개발자',
  'grades': ['SENIOR', 'EXPERT'],
  'skills': ['Java', 'Spring'],
  'count': 1,
  'search_query': 'Java 및 Spring 기반 백엔드 개발 역량을 갖춘 고급 또는 특급 수준의 PL급 개발자'},
 {'position': '개발자',
  'role': '프론트',
  'grades': ['INTERMEDIATE'],
  'skills': ['React'],
  'count': 2,
  'search_query': 'React 활용 능력을 갖춘 중급 수준의 프론트엔드 개발자'}]

In [7]:
# 포지션별 벡터 검색 (등급 필터 + 유사도 검색)
results_by_position = {}

for pos in positions:
    grade_filter = " OR ".join([f"grade = '{g}'" for g in pos["grades"]])
    filter_expr = f"status = 'AVAILABLE' AND ({grade_filter})"

    results = search_service.search(
        query=pos["search_query"],
        top_k=pos["count"] + 2,
        filter=filter_expr,
    )
    results_by_position[pos["position"]] = {"requirement": pos, "candidates": results}

# 결과 출력
for position, data in results_by_position.items():
    req = data["requirement"]
    print(f"\n{'='*60}")
    print(f"[{position}] {req['role']} | 등급: {req['grades']} | 스킬: {req['skills']} | {req['count']}명 필요")
    print(f"{'='*60}")
    for r in data["candidates"]:
        print(f"  score: {r.score:.4f} | {r.metadata['engineer_id']} | {r.metadata['grade']} | {r.metadata['engineer_role']}")
        print(f"  {r.document.page_content[:100].strip()}...")
        print()


[PL] 백엔드 개발자 | 등급: ['SENIOR', 'EXPERT'] | 스킬: ['Java', 'Spring'] | 1명 필요
  score: 0.8817 | eng-001 | SENIOR | DEVELOPER
  [기술스택]
        Java / Spring Boot / PostgreSQL / Redis / Docker
        
        [경력]
        8년차 백엔...

  score: 0.8817 | eng-001 | SENIOR | DEVELOPER
  [기술스택]
        Java / Spring Boot / PostgreSQL / Redis / Docker
        
        [경력]
        8년차 백엔...

  score: 0.8817 | eng-001 | SENIOR | DEVELOPER
  [기술스택]
        Java / Spring Boot / PostgreSQL / Redis / Docker
        
        [경력]
        8년차 백엔...


[개발자] 프론트 | 등급: ['INTERMEDIATE'] | 스킬: ['React'] | 2명 필요
  score: 0.8712 | eng-004 | INTERMEDIATE | DEVELOPER
  [기술스택]
        React / JavaScript / Vue.js / Chart.js / CSS
        
        [경력]
        4년차 프론트엔드...

  score: 0.8712 | eng-004 | INTERMEDIATE | DEVELOPER
  [기술스택]
        React / JavaScript / Vue.js / Chart.js / CSS
        
        [경력]
        4년차 프론트엔드...

  score: 0.8712 | eng-004 | INTERMEDIATE | DEVELOPER
  [기술스택]
        React / JavaScript

In [10]:
from IPython.display import Markdown

# LLM 최종 추천
context_parts = []
for position, data in results_by_position.items():
    req = data["requirement"]
    candidates_text = "\n".join([
        f"- {r.metadata['engineer_id']} (등급: {r.metadata['grade']}, 유사도: {r.score:.4f})\n{r.document.page_content.strip()}"
        for r in data["candidates"]
    ])
    context_parts.append(f"[{position} - {req['role']}]\n{candidates_text}")

context = "\n\n".join(context_parts)

recommend_prompt = f"""
아래 프로젝트 요구사항과 후보 엔지니어 프로필을 보고, 각 포지션에 가장 적합한 엔지니어를 추천하세요.

프로젝트 요구사항:
{project_requirement}

후보 엔지니어 프로필:
{context}

각 포지션별로 최적 후보를 추천하고, 추천 이유를 간략히 설명하세요.
"""

response = llm.invoke(recommend_prompt)
content = response.content
if isinstance(content, list):
    content = "".join(block if isinstance(block, str) else block.get("text", "") for block in content)
Markdown(content)

제공해주신 프로젝트 요구사항과 후보자 프로필을 분석하여, 가장 적합한 엔지니어를 다음과 같이 추천합니다.

---

### **[추천 결과]**

| 포지션 | 인원 | 추천 후보 | 등급 |
| :--- | :---: | :--- | :--- |
| **PL (백엔드)** | 1명 | **eng-001** | 고급 (Senior) |
| **개발자 (프론트)** | 2명 | **eng-004** (2명 배치) | 중급 (Intermediate) |

---

### **[추천 사유]**

#### **1. PL (백엔드): eng-001**
*   **적합성:** 프로젝트에서 요구하는 '고급/특급' 등급과 'Java/Spring' 스킬셋을 완벽히 충족합니다.
*   **전문성:** 8년차 경력으로 현대모비스, LG CNS, 삼성SDS 등 대형 엔터프라이즈 환경에서의 ERP 및 MES 구축 경험이 풍부합니다. 특히 마이그레이션 프로젝트에서 필수적인 **데이터 처리 최적화(쿼리 튜닝)** 및 **시스템 자동화(Spring Batch)** 역량을 보유하고 있어, ITSM 시스템 이전 프로젝트의 리딩 역할을 수행하는 데 매우 적합합니다.

#### **2. 개발자 (프론트): eng-004**
*   **적합성:** 프로젝트에서 요구하는 '중급' 등급과 'React' 기술 스택을 보유하고 있습니다.
*   **전문성:** 포스코, 네이버 등 대규모 프로젝트에서 대시보드 및 리포트 화면을 개발한 경험이 있어, 복잡한 비즈니스 로직을 시각화해야 하는 ITSM 서비스의 UI/UX 구현에 강점이 있습니다. Chart.js 등의 활용 경험은 ITSM 시스템의 지표 관리 화면 구현에 큰 도움이 될 것으로 판단됩니다.

---

**[참고 사항]**
*   현재 제공된 후보자 풀에 동일한 ID(eng-001, eng-004)가 중복 기재되어 있으나, 해당 후보자들이 요구사항의 등급과 기술 스택을 가장 잘 부합하고 있으므로 해당 인원들을 각 포지션에 투입하는 것이 최선입니다.
*   프론트엔드 포지션은 총 2명이 필요하므로, 동일한 역량을 갖춘 **eng-004 후보자 2명을 배정**하여 개발 속도와 코드 품질의 일관성을 유지하는 것을 권장합니다.